# Demo - Model Monitoring

In this demo, we will show you how to monitor the performance of a machine learning model using Databricks. We will use a diabetes dataset to train a model, track inference data, and analyze its performance using Databricks' built-in features. Additionally, we will detect drift and demonstrate how to handle model retraining and continuous monitoring.

## Learning Objectives

By the end of this demo, you will be able to:

* Train and analyze a machine learning model's inference logs.
* Monitor the model's performance and detect anomalies or drift.
* Handle drift detection and trigger retraining when needed.
* Utilize Databricks Lakehouse Monitoring to continuously track and alert on model performance metrics.

## Requirements

Please review the following requirements before starting the lesson:

* To run this notebook, you need to use one of the following Databricks runtime(s): **`15.3.x-cpu-ml-scala2.12`**



## Classroom Setup

Before starting the demo, run the provided classroom setup script. This script will define configuration variables necessary for the demo. Execute the following cell:

In [0]:
%pip install "databricks-sdk>=0.28.0"

dbutils.library.restartPython()

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:

#%run ../Includes/Classroom-Setup-3.1

# Prepare Model

## Load dataset

In this section, we load the dataset for diabetes classification. Since the dataset is small and we want to go straight to training a classic model, we load it directly with Pandas.

https://www.kaggle.com/datasets/iammustafatz/diabetes-prediction-dataset/data

In [0]:
import os


In [0]:
import pandas as pd
from pyspark.sql.functions import lit, col, from_unixtime, array
from pyspark.sql.types import DoubleType
import time
import json

# 1. URL pública válida contendo o mesmo arquivo 'diabetes_prediction_dataset.csv'
csv_url = "https://raw.githubusercontent.com/NANITH777/Diabetes-Prediction-ID3_Alg-ML-Models/main/diabetes_prediction_dataset.csv"

# 2. Carrega o CSV via Pandas
df_raw = pd.read_csv(csv_url)


# 4. Converte de Pandas para Spark DataFrame e salva no formato Delta
spark_df = spark.createDataFrame(df_raw)


In [0]:
spark_df.show(5)

+------+----+------------+-------------+---------------+-----+-----------+-------------------+--------+
|gender| age|hypertension|heart_disease|smoking_history|  bmi|HbA1c_level|blood_glucose_level|diabetes|
+------+----+------------+-------------+---------------+-----+-----------+-------------------+--------+
|Female|80.0|           0|            1|          never|25.19|        6.6|                140|       0|
|Female|54.0|           0|            0|        No Info|27.32|        6.6|                 80|       0|
|  Male|28.0|           0|            0|          never|27.32|        5.7|                158|       0|
|Female|36.0|           0|            0|        current|23.45|        5.0|                155|       0|
|  Male|76.0|           1|            1|        current|20.14|        4.8|                155|       0|
+------+----+------------+-------------+---------------+-----+-----------+-------------------+--------+
only showing top 5 rows


In [0]:
# Convert to Pandas DataFrame
diabetes_df_pd = spark_df.toPandas()

display(diabetes_df_pd.head())

gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
Female,80.0,0,1,never,25.19,6.6,140,0
Female,54.0,0,0,No Info,27.32,6.6,80,0
Male,28.0,0,0,never,27.32,5.7,158,0
Female,36.0,0,0,current,23.45,5.0,155,0
Male,76.0,1,1,current,20.14,4.8,155,0


## Train / New Requests Split

In a typical model training scenario, we would split the data into **training** and **test** sets to evaluate the model's performance. However, our focus here is on what happens once a model is trained and integrated into a production environment, where it encounters new data that was not part of its original training set.

To simulate this situation, we will split our existing dataset into a **training set** and a **new requests** set. This separation allows us to explore how the model handles incoming data that might differ from its training data, which can help us identify potential issues such as drift.

In order to simulate a drift in the model features, we will use the `Age` feature of our data to divide the sets. This approach enables us to study how the model's predictions change when exposed to data with characteristics that differ from the original training data.

In [0]:
diabetes_df_pd['diabetes'].unique()

array([0, 1])

0: Não possui diabetes (ou tem risco/incidência nula).

1: Possui diabetes (ou foi diagnosticada com a condição).

In [0]:
diabetes_df_pd['Diabetes_binary'] = diabetes_df_pd['diabetes']



In [0]:
diabetes_df_pd = diabetes_df_pd.drop(columns=['diabetes'])

In [0]:
diabetes_df_pd['id'] = diabetes_df_pd.index


In [0]:
# Split the data into training and request sets based on the 'Age' feature
train_df = diabetes_df_pd[diabetes_df_pd['age'] <= 9]
request_df = diabetes_df_pd[diabetes_df_pd['age'] > 9]

# Define the target column
target_col = "Diabetes_binary"

# Prepare training features and labels
X_train = train_df.drop(labels=[target_col, 'id'], axis=1)
y_train = train_df[target_col]

# Prepare request features and labels
X_request = request_df.drop(labels=[target_col], axis=1)
y_request = request_df[target_col]

## Fit a Classification Model

Let's go ahead and fit a Decision Tree model and register it with Unity Catalog.

In [0]:
import mlflow
from sklearn.tree import DecisionTreeClassifier
from mlflow.models.signature import infer_signature
from sklearn.preprocessing import LabelEncoder

# Set the MLflow registry URI to Unity Catalog
mlflow.set_registry_uri("databricks-uc")

# Enable automatic logging with MLflow
mlflow.sklearn.autolog(log_input_examples=True)

# Encode categorical columns
X_train_encoded = X_train.copy()
categorical_cols = ['gender', 'smoking_history']

# adaptar o texto para números 
for col in categorical_cols:
    le = LabelEncoder()
    X_train_encoded[col] = le.fit_transform(X_train_encoded[col])

# Initialize and train the Decision Tree Classifier
dtc = DecisionTreeClassifier()
dtc_mdl = dtc.fit(X_train_encoded, y_train)

catalog_name = "workspace"
schema_name = "default"

# Define the model name using catalog and schema
model_name = f"{catalog_name}.{schema_name}.diabetes_model"

# Infer the model signature
signature = infer_signature(X_train_encoded, y_train)

# Log the model to MLflow
mlflow.sklearn.log_model(
    sk_model=dtc_mdl,
    artifact_path="model-artifacts",
    signature=signature,
    registered_model_name=model_name
)

print(f"Model '{model_name}' has been registered with MLflow.")

2026/08/09 23:52:10 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'b22273e801cd46b289b416b7a93e0699', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/08/09 23:52:10 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values 

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

🔗 Created version '10' of model 'workspace.default.diabetes_model': https://dbc-0b69ef93-5247.cloud.databricks.com/explore/data/models/workspace/default/diabetes_model/version/10?o=7474657872577658


Model 'workspace.default.diabetes_model' has been registered with MLflow.


## Save the Training Data as Reference for Drift

We can save the training dataset that was used to train the model. This can be used later during monitoring to provide a reference to determine if a drift has happened between training and the new incoming requests.

In [0]:
from pyspark.sql.functions import lit, col
from pyspark.sql.types import DoubleType

# Create the Spark DataFrame and rename 'Diabetes_binary' to 'labeled_data', while casting it to DoubleType
spark_df = (spark.createDataFrame(train_df)
            .withColumn('model_id', lit(0))
            .withColumn('labeled_data', col('Diabetes_binary').cast(DoubleType()).alias('labeled_data')))

# Write the DataFrame to Delta format
(
    spark_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .option("delta.enableChangeDataFeed", "true")
    .saveAsTable(f"{catalog_name}.{schema_name}.baseline_features")
)

# Processing Inference Table Data

In this section, we focus on extracting and analyzing data logged in the **inference table**. This table contains detailed information on each request and response received by the model. However, the raw format of this data is optimized for storage rather than immediate analysis. To effectively monitor and interpret the data, we will convert it into a more analyzable format using a series of steps.

**Steps to Process Inference Table Data:**

1. **Timestamp Conversion:** Convert the timestamp data from milliseconds to a human-readable timestamp format. This adjustment helps in analyzing the data based on when events occurred.
2. **Unpacking JSON:** Requests and responses in the inference table are stored in JSON format. We'll unpack these JSON strings into a structured DataFrame format, making it easier to work with the data for analysis.
3. **Exploding Batched Requests:** If the model receives batched requests, these will be exploded into individual records for simplified analysis. Each entry in a batch request is treated as a separate data point to ensure accuracy.
4. **Schema Transformation:** We will transform the schema of the extracted data to better align it with analytical needs, facilitating easier data interpretation and monitoring.

These steps will ensure that the inference data is optimized for analysis and monitoring, providing clear insights into how the model is performing over time.

---

> **Note for Instructors:** The **inference table** used in this demo contains simulated or pre-populated data that mimics the behavior of a live inference table receiving real-time requests. This allows students to focus on analyzing and processing the data without needing to deploy a model during the exercise. It is important to guide students on how to handle live inference data in a real-world scenario where new requests are logged continuously.
>
> **Please note that:** Typically, we need to wait for the inference table to populate with new data, which usually takes about 5-7 minutes after batch predictions are triggered.

In [0]:
spark_df.show(2)

+------+---+------------+-------------+---------------+-----+-----------+-------------------+---------------+---+--------+------------+
|gender|age|hypertension|heart_disease|smoking_history|  bmi|HbA1c_level|blood_glucose_level|Diabetes_binary| id|model_id|labeled_data|
+------+---+------------+-------------+---------------+-----+-----------+-------------------+---------------+---+--------+------------+
|  Male|5.0|           0|            0|        No Info| 18.8|        6.2|                 85|              0| 21|       0|         0.0|
|Female|4.0|           0|            0|        No Info|13.99|        4.0|                140|              0| 24|       0|         0.0|
+------+---+------------+-------------+---------------+-----+-----------+-------------------+---------------+---+--------+------------+
only showing top 2 rows


In [0]:
# Read and display the inference table
try:
    inference_df = spark.read.table(f"{catalog_name}.{schema_name}.model_inference_table")
    
    if inference_df.count() > 0:
        display(inference_df)
    else:
        print("The inference table is empty.")
except Exception as e:
    print(f"Error finding the table: {e}")

{"ts": "2026-08-09 23:52:44.090", "level": "ERROR", "logger": "pyspark.sql.connect.logging", "msg": "GRPC Error received", "context": {}, "exception": {"class": "_MultiThreadedRendezvous", "msg": "<_MultiThreadedRendezvous of RPC that terminated with:\n\tstatus = StatusCode.INTERNAL\n\tdetails = \"[TABLE_OR_VIEW_NOT_FOUND] The table or view `workspace`.`default`.`model_inference_table` cannot be found. Verify the spelling and correctness of the schema and catalog.\nSearch path: [`system`.`session`, `system`.`builtin`, `system`.`ai`, `workspace`.`default`].\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01;\n'Aggregate [unresolvedalias(count(1))]\n+- 'UnresolvedRelation [workspace, default, model_inference_table], [], false\n\"\n\tdebug_error_string = \"UNKNOWN:Error received from peer  {grpc_message:\"

Error finding the table: [TABLE_OR_VIEW_NOT_FOUND] The table or view `workspace`.`default`.`model_inference_table` cannot be found. Verify the spelling and correctness of the schema and catalog.
Search path: [`system`.`session`, `system`.`builtin`, `system`.`ai`, `workspace`.`default`].
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01;
'Aggregate [unresolvedalias(count(1))]
+- 'UnresolvedRelation [workspace, default, model_inference_table], [], false


JVM stacktrace:
org.apache.spark.sql.catalyst.ExtendedAnalysisException
	at org.apache.spark.sql.errors.QueryCompilationErrors$.tableOrViewNotFoundWithSearchPath(QueryCompilationErrors.scala:1464)
	at org.apache.spark.sql.catalyst.analysis.package$AnalysisErrorAt.tableNotFound(package.scala:97)
	at org.apache.spark.sql.catalyst.analysis.RelationResolution

## Conversion Helper Functions

The JSON fields within the inference table require transformation into structured columns. We will define helper functions to handle this conversion efficiently.

# simular o test dataframe

In [0]:
import json
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, ArrayType, DoubleType
# simular para o teste 
# Create simulated inference data with 2 rows
inference_data = [
    {
        "timestamp_ms": 1723000000000,  # Unix timestamp in milliseconds
        "status_code": 200,
        "request": json.dumps([{
            "gender": 1.0,
            "age": 10.0,
            "hypertension": 0,
            "heart_disease": 0,
            "smoking_history": 0.0,
            "bmi": 27.32,
            "HbA1c_level": 4.0,
            "blood_glucose_level": 80,
            "Diabetes_binary": 0,
            "id": 90000
        }]),
        "response": {
            "predictions": [0]
        }
    },
    {
        "timestamp_ms": 1723100000000,
        "status_code": 200,
        "request": json.dumps([{
            "gender": 0.0,
            "age": 12.0,
            "hypertension": 1,
            "heart_disease": 0,
            "smoking_history": 2.0,
            "bmi": 32.5,
            "HbA1c_level": 6.5,
            "blood_glucose_level": 140,
            "Diabetes_binary": 1,
            "id": 90001
        }]),
        "response": {
            "predictions": [1]
        }
    }
]

# Define schema for the inference DataFrame
inference_schema = StructType([
    StructField("timestamp_ms", LongType(), True),
    StructField("status_code", IntegerType(), True),
    StructField("request", StringType(), True),
    StructField("response", StructType([
        StructField("predictions", ArrayType(IntegerType()), True)
    ]), True)
])

# Create the simulated inference DataFrame
inference_df = spark.createDataFrame(inference_data, schema=inference_schema)

# Display to verify
print("Simulated inference_df created with 2 rows:")
display(inference_df)

Simulated inference_df created with 2 rows:


timestamp_ms,status_code,request,response
1723000000000,200,"[{""gender"": 1.0, ""age"": 10.0, ""hypertension"": 0, ""heart_disease"": 0, ""smoking_history"": 0.0, ""bmi"": 27.32, ""HbA1c_level"": 4.0, ""blood_glucose_level"": 80, ""Diabetes_binary"": 0, ""id"": 90000}]",List(List(0))
1723100000000,200,"[{""gender"": 0.0, ""age"": 12.0, ""hypertension"": 1, ""heart_disease"": 0, ""smoking_history"": 2.0, ""bmi"": 32.5, ""HbA1c_level"": 6.5, ""blood_glucose_level"": 140, ""Diabetes_binary"": 1, ""id"": 90001}]",List(List(1))


In [0]:
# Read and display the inference table
try:
    #inference_df = spark.read.table(f"{catalog_name}.{schema_name}.model_inference_table")
    display(inference_df.tail())
    
    if inference_df.count() > 0:
        display(inference_df)
    else:
        print("The inference table is empty.")
except Exception as e:
    print(f"Error finding the table: {e}")

Error finding the table: DataFrame.tail() missing 1 required positional argument: 'num'


In [0]:
from pyspark.sql import DataFrame, functions as F, types as T
import json
import pandas as pd


# Define a UDF to handle JSON unpacking
def convert_to_record_json(json_str: str) -> str:
    """
    Converts records from different accepted JSON formats into a common, record-oriented
    DataFrame format which can be parsed by the PySpark function `from_json`.

    :param json_str: The JSON string containing the request or response payload.
    :return: A JSON string containing the converted payload in a record-oriented format.


        Standardizes payload JSON formats (dataframe_records, dataframe_split,
    instances, inputs, predictions) into a uniform record-oriented JSON string.
    """
    try:
        # Attempt to parse the JSON string
        request = json.loads(json_str)
    except json.JSONDecodeError:
        # If parsing fails, return the original string
        return json_str

    output = []
    if isinstance(request, dict):
        # Handle different JSON formats and convert to a common format

        if "dataframe_records" in request:
            output.extend(request["dataframe_records"])

        elif "dataframe_split" in request:
            dataframe_split = request["dataframe_split"]
            output.extend([dict(zip(dataframe_split["columns"], values)) for values in dataframe_split["data"]])

        elif "instances" in request:
            output.extend(request["instances"])
            
        elif "inputs" in request:
             output.extend([dict(zip(request["inputs"], values)) for values in  zip(*request["inputs"].values())])

                
        elif "predictions" in request:
            preds = data["predictions"]
            output.extend([{"predictions": p} for p in preds])     

        return json.dumps(output)    

    else: 
        # If the format is unsupported, return the original string
        return json_str



# Pandas UDF for batch processing inference logs
@F.pandas_udf(T.StringType())
def json_consolidation_udf(json_strs: pd.Series) -> pd.Series:
    """
    Applies the JSON conversion logic over a Pandas Series column in PySpark.
    """
    return json_strs.apply(convert_to_record_json)


## Processing the Raw Inference Data

We will process the inference data by unpacking JSON fields and converting relevant data into scalar values.

In [0]:
from pyspark.sql import DataFrame, functions as F, types as T
from pyspark.sql.types import TimestampType

def process_requests(requests_raw: DataFrame) -> DataFrame:
    """
    Processes a stream of raw requests and:
      - Unpacks JSON payloads for requests
      - Extracts relevant features as scalar values (first element of each array)
      - Converts Unix epoch millisecond timestamps to Spark TimestampType
    """
    # Calculate the current timestamp in seconds
    current_ts = int(spark.sql("SELECT unix_timestamp(current_timestamp())").collect()[0][0])

    # Define the start timestamp for 30 days ago
    start_ts = current_ts - 30 * 24 * 60 * 60  # 30 days in seconds

    # Dynamically calculate the min and max values of timestamp_ms
    # Calcula dinamicamente o valor mínimo e máximo de 'timestamp_ms' no DataFrame bruto
    min_max = requests_raw.agg(
        F.min("timestamp_ms").alias("min_ts"),
        F.max("timestamp_ms").alias("max_ts")
    ).collect()[0]
    
    # Converte os limites obtidos de milissegundos para segundos
    min_ts = min_max["min_ts"] / 1000  # Convert from milliseconds to seconds
    max_ts = min_max["max_ts"] / 1000  # Convert from milliseconds to seconds

    # Transform timestamp_ms to span the last month
    requests_timestamped = requests_raw.withColumn(
        'timestamp',
        (start_ts + ((F.col("timestamp_ms") / 1000 - min_ts) / (max_ts - min_ts)) * (current_ts - start_ts)).cast(TimestampType())
    ).drop("timestamp_ms")

    # Unpack JSON for the 'request' column only, since 'response' is already structured
    requests_unpacked = requests_timestamped \
        .withColumn("request", json_consolidation_udf(F.col("request"))) \
        .withColumn('request', F.from_json(F.col("request"), F.schema_of_json(
           # '[{"HighBP": 1.0, "HighChol": 0.0, "CholCheck": 1.0, "BMI": 26.0, "Smoker": 0.0, "Stroke": 0.0, "HeartDiseaseorAttack": 0.0, "PhysActivity": 1.0, "Fruits": 0.0, "Veggies": 1.0, "HvyAlcoholConsump": 0.0, "AnyHealthcare": 1.0, "NoDocbcCost": 0.0, "GenHlth": 3.0, "MentHlth": 5.0, "PhysHlth": 30.0, "DiffWalk": 0.0, "Sex": 1.0, "Age": 4.0, "Education": 6.0, "Income": 8.0, "id": 1}]')))
            '[{"gender": 1.0, "age": 4.0, "hypertension": 0, "heart_disease": 0, "smoking_history": 0.0, "bmi": 30.0, "HbA1c_level": 5.0, "blood_glucose_level": 80, "id": 100000}]')))
    # Extract feature columns as scalar values (first element of each array) # ajustar colunas corretas
    #feature_columns = ["HighBP", "HighChol", "CholCheck", "BMI", "Smoker", "Stroke", "HeartDiseaseorAttack",
    #                   "PhysActivity", "Fruits", "Veggies", "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost",
    #                   "GenHlth", "MentHlth", "PhysHlth", "DiffWalk", "Sex", "Age", "Education", "Income", "id"]

    # F.schema_of_json infere o schema a partir do exemplo:
    #    Schema: Array<Struct<
    #    gender: double,
    #    age: double,
    #    hypertension: integer,
    #    heart_disease: integer,
    #    smoking_history: double,
    #    bmi: double,
    #    HbA1c_level: double,
    #    blood_glucose_level: integer,
    #    id: integer
    #    >>
    # F.from_json usa esse schema para fazer o parse:
    #request (coluna) → Array[Struct] com campos acessíveis
    feature_columns = ['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history',
       'bmi', 'HbA1c_level', 'blood_glucose_level', 'id']
    for col_name in feature_columns:
        # Extract the first element of each array for all feature columns
        requests_unpacked = requests_unpacked.withColumn(col_name, F.col(f"request.{col_name}")[0])

    # Extract predictions from the 'response' column without using from_json
    requests_unpacked = requests_unpacked.withColumn("Diabetes_binary", F.col("response.predictions")[0])
    
    # Drop unnecessary columns and add model_id
    requests_cleaned = requests_unpacked.drop("request", "response").withColumn("model_id", F.lit(0).cast(T.IntegerType()))
    
    final_df = requests_cleaned
    
    return final_df

In [0]:
from pyspark.sql import DataFrame, functions as F, types as T
from pyspark.sql.types import TimestampType

def process_requests_old(requests_raw: DataFrame) -> DataFrame:
    """
    Processes a stream of raw requests and:
      - Unpacks JSON payloads for requests
      - Extracts relevant features as scalar values (first element of each array)
      - Converts Unix epoch millisecond timestamps to Spark TimestampType
    """
    # Calculate the current timestamp in seconds
    current_ts = int(spark.sql("SELECT unix_timestamp(current_timestamp())").collect()[0][0])

    # Define the start timestamp for 30 days ago
    start_ts = current_ts - 30 * 24 * 60 * 60  # 30 days in seconds

    # Dynamically calculate the min and max values of timestamp_ms
    # Calcula dinamicamente o valor mínimo e máximo de 'timestamp_ms' no DataFrame bruto
    min_max = requests_raw.agg(
        F.min("timestamp_ms").alias("min_ts"),
        F.max("timestamp_ms").alias("max_ts")
    ).collect()[0]
    
    # Converte os limites obtidos de milissegundos para segundos
    min_ts = min_max["min_ts"] / 1000  # Convert from milliseconds to seconds
    max_ts = min_max["max_ts"] / 1000  # Convert from milliseconds to seconds

    # Transform timestamp_ms to span the last month
    requests_timestamped = requests_raw.withColumn(
        'timestamp',
        (start_ts + ((F.col("timestamp_ms") / 1000 - min_ts) / (max_ts - min_ts)) * (current_ts - start_ts)).cast(TimestampType())
    ).drop("timestamp_ms")

    # Unpack JSON for the 'request' column only, since 'response' is already structured
    requests_unpacked = requests_timestamped \
        .withColumn("request", json_consolidation_udf(F.col("request"))) \
        .withColumn('request', F.from_json(F.col("request"), F.schema_of_json(
           # '[{"HighBP": 1.0, "HighChol": 0.0, "CholCheck": 1.0, "BMI": 26.0, "Smoker": 0.0, "Stroke": 0.0, "HeartDiseaseorAttack": 0.0, "PhysActivity": 1.0, "Fruits": 0.0, "Veggies": 1.0, "HvyAlcoholConsump": 0.0, "AnyHealthcare": 1.0, "NoDocbcCost": 0.0, "GenHlth": 3.0, "MentHlth": 5.0, "PhysHlth": 30.0, "DiffWalk": 0.0, "Sex": 1.0, "Age": 4.0, "Education": 6.0, "Income": 8.0, "id": 1}]')))
            '[{"gender": 1.0, "age": 4.0, "hypertension": 0, "heart_disease": 0, "smoking_history": 0.0, "bmi": 30.0, "HbA1c_level": 5.0, "blood_glucose_level": 80, "id": 100000}]')))
    # Extract feature columns as scalar values (first element of each array) # ajustar colunas corretas
    #feature_columns = ["HighBP", "HighChol", "CholCheck", "BMI", "Smoker", "Stroke", "HeartDiseaseorAttack",
    #                   "PhysActivity", "Fruits", "Veggies", "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost",
    #                   "GenHlth", "MentHlth", "PhysHlth", "DiffWalk", "Sex", "Age", "Education", "Income", "id"]
    feature_columns = ['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history',
       'bmi', 'HbA1c_level', 'blood_glucose_level', 'Diabetes_binary', 'id']
    for col_name in feature_columns:
        # Extract the first element of each array for all feature columns
        requests_unpacked = requests_unpacked.withColumn(col_name, F.col(f"request.{col_name}")[0])

    # Extract predictions from the 'response' column without using from_json
    requests_unpacked = requests_unpacked.withColumn("Diabetes_binary", F.col("response.predictions")[0])
    
    # Drop unnecessary columns and add model_id
    requests_cleaned = requests_unpacked.drop("request", "response").withColumn("model_id", F.lit(0).cast(T.IntegerType()))
    
    final_df = requests_cleaned
    
    return final_df

In [0]:
diabetes_df_pd.columns

Index(['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history',
       'bmi', 'HbA1c_level', 'blood_glucose_level', 'Diabetes_binary', 'id'],
      dtype='object')

In [0]:
diabetes_df_pd.tail()

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,Diabetes_binary,id
99995,Female,80.0,0,0,No Info,27.32,6.2,90,0,99995
99996,Female,2.0,0,0,No Info,17.37,6.5,100,0,99996
99997,Male,66.0,0,0,former,27.83,5.7,155,0,99997
99998,Female,24.0,0,0,never,35.42,4.0,100,0,99998
99999,Female,57.0,0,0,current,22.43,6.6,90,0,99999


# Analyzing Processed Requests

After processing and unpacking the logged data from the inference table, the next step is to analyze the successfully answered requests by the model. This involves filtering the data to focus on successful interactions, merging with additional information, and thoroughly examining the model's performance.

**Steps for Analyzing Processed Requests:**

1. **Filtering Requests:** Initially, we filter out the requests to only include those with a successful status code (200). This ensures that we are analyzing only the requests where the model was able to generate predictions.
2. **Displaying Logs:** The filtered logs are then displayed, providing a clear view of the processed requests and their outcomes.
3. **Merging Data:** We also merge these logs with additional label data that categorizes the results. This step is crucial for evaluating the model's performance against known outcomes.
4. **Final Display:** The merged DataFrame is displayed, showing the complete information of the requests along with their corresponding labels. This provides a full picture of how the model is performing in real-world scenarios.

This detailed view helps in understanding the effectiveness of the model and in making necessary adjustments based on real-world data feedback.

In [0]:
# Check if inference_df has data before processing
try:
    if inference_df.count() == 0:
        print("No inference data available. The inference table is empty.")
        print("To proceed, you need to either:")
        print("1. Create and populate the inference table with sample data, or")
        print("2. Deploy the model to a serving endpoint and generate inference requests")
    else:
        # Apply the function to the inference DataFrame
        model_logs_df = process_requests(inference_df.where("status_code = 200"))
        #model_logs_df = process_requests(inference_df)  #simular 
       
        # Display the updated DataFrame to confirm
        model_logs_df.display()
except Exception as e:
    print(f"Error: The inference table 'workspace.default.model_inference_table' cannot be found.")
    print("To proceed, you need to either:")
    print("1. Create and populate the inference table 'workspace.default.model_inference_table' with sample data, or")
    print("2. Deploy the model to a serving endpoint with inference table logging enabled")

status_code,timestamp,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,id,Diabetes_binary,model_id
200,2026-07-10T23:52:47.000Z,1.0,10.0,0,0,0.0,27.32,4.0,80,90000,0,0
200,2026-08-09T23:52:47.000Z,0.0,12.0,1,0,2.0,32.5,6.5,140,90001,1,0


In [0]:
# Check if inference_df has data before processing
try:
    if inference_df.count() == 0:
        print("No inference data available. The inference table is empty.")
        print("To proceed, you need to either:")
        print("1. Create and populate the inference table with sample data, or")
        print("2. Deploy the model to a serving endpoint and generate inference requests")
    else:
        # Apply the function to the inference DataFrame
        model_logs_df = process_requests(inference_df.where("status_code = 200"))
       
        # Display the updated DataFrame to confirm
        model_logs_df.display()
except Exception as e:
    print(f"Error processing inference data: {e}")
    print("To proceed, you need to either:")
    print("1. Create and populate the inference table 'workspace.default.model_inference_table' with sample data, or")
    print("2. Deploy the model to a serving endpoint with inference table logging enabled")

status_code,timestamp,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,id,Diabetes_binary,model_id
200,2026-07-10T23:53:01.000Z,1.0,10.0,0,0,0.0,27.32,4.0,80,90000,0,0
200,2026-08-09T23:53:01.000Z,0.0,12.0,1,0,2.0,32.5,6.5,140,90001,1,0


In [0]:
# Check if inference_df has data before processing
try:
    if inference_df.count() == 0:
        print("No inference data available. The inference table is empty.")
        print("To proceed, you need to either:")
        print("1. Create and populate the inference table with sample data, or")
        print("2. Deploy the model to a serving endpoint and generate inference requests")
    else:
        # Apply the function to the inference DataFrame
        model_logs_df = process_requests(inference_df.where("status_code = 200"))
        #model_logs_df = process_requests(inference_df)  #simular 
       
        # Display the updated DataFrame to confirm
        model_logs_df.display()
except Exception as e:
    print(f"Error: The inference table 'workspace.default.model_inference_table' cannot be found.")
    print("To proceed, you need to either:")
    print("1. Create and populate the inference table 'workspace.default.model_inference_table' with sample data, or")
    print("2. Deploy the model to a serving endpoint with inference table logging enabled")

status_code,timestamp,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,id,Diabetes_binary,model_id
200,2026-07-10T23:53:03.000Z,1.0,10.0,0,0,0.0,27.32,4.0,80,90000,0,0
200,2026-08-09T23:53:03.000Z,0.0,12.0,1,0,2.0,32.5,6.5,140,90001,1,0


In [0]:
# Check if model_logs_df exists before proceeding
try:
    # Rename the column in the pandas DataFrame and convert it to a Spark DataFrame
    label_pd_df = diabetes_df_pd.rename(columns={'Diabetes_binary': 'labeled_data'})
    label_pd_df = spark.createDataFrame(label_pd_df)
    
    # Perform the join operation
    model_logs_df_labeled = model_logs_df.join(
        label_pd_df.select("id", "labeled_data"),
        on=["id"],
        how="left"
    ).drop("id")
    
    # Display the result
    display(model_logs_df_labeled)
except NameError:
    print("Cannot proceed: model_logs_df is not available.")
    print("This cell requires the inference data to be processed in the previous cell.")
    print("Please ensure the inference table exists and contains data before running this cell.")

status_code,timestamp,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,Diabetes_binary,model_id,labeled_data
200,2026-07-10T23:53:03.000Z,1.0,10.0,0,0,0.0,27.32,4.0,80,0,0,0
200,2026-08-09T23:53:03.000Z,0.0,12.0,1,0,2.0,32.5,6.5,140,1,0,0


# Persisting Processed Model Logs

After analyzing the model's responses and merging them with relevant labels, the final step involves saving these enriched logs for long-term monitoring and analysis. This step is critical for maintaining a historical record of model performance and enabling further analytical studies.

**Steps for Saving Model Logs:**

1. **Preparing the Data:** The processed and labeled model logs are prepared for storage. At this point, the data includes all necessary details, such as predictions, request metadata, and corresponding labels.
2. **Setting the Storage Mode:** We use the append mode when saving the DataFrame. This approach ensures that new entries are added to the existing dataset without overwriting previous logs, thereby accumulating a comprehensive log over time.
3. **Saving the DataFrame:** The logs are saved as a table in the Databricks catalog under the specified catalog name and schema. Organizing data in this structured manner facilitates efficient management and easy retrieval for future analysis and monitoring.
4. **Confirming Save Operation:** Finally, the operation concludes with the confirmation that the logs have been successfully appended to the designated table, indicating successful data persistence.

By systematically saving these logs, we establish a robust foundation for ongoing monitoring of the model's performance. This enables proactive management, detailed analysis, and continuous optimization of machine learning operations.

In [0]:
model_logs_df_labeled.write.mode("append").saveAsTable(f'{catalog_name}.{schema_name}.model_logs')

For efficient execution, enable CDF so monitoring can incrementally process the data.

In [0]:
spark.sql(f'ALTER TABLE {catalog_name}.{schema_name}.model_logs SET TBLPROPERTIES (delta.enableChangeDataFeed = true)')

DataFrame[]

# Creating an Inference Monitor with Databricks Lakehouse Monitoring

Once your model logs are saved and structured for analysis, the next essential step is setting up monitoring to continuously track model performance and detect any anomalies or drift in real-time. You can set up an inference monitor using Databricks Lakehouse Monitoring through two approaches. Both methods will enable you to monitor your model's performance efficiently, ensuring that any necessary adjustments or retraining can be handled promptly.

### Option 1: Using the Notebook

For those who are comfortable with scripting and want more control over the monitoring setup, you can use the provided notebook commands to configure and initiate the monitoring of your model logs. This method allows you to automate and customize the monitoring according to specific needs and thresholds.

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.catalog import MonitorInferenceLog, MonitorInferenceLogProblemType, MonitorInfoStatus, MonitorRefreshInfoState, MonitorMetric

w = WorkspaceClient()
table_name = f'{catalog_name}.{schema_name}.model_logs'
baseline_table_name = f"{catalog_name}.{schema_name}.baseline_features"

In [0]:
help(w.quality_monitors.create)

Help on method create in module databricks.sdk.service.catalog:

create(table_name: 'str', output_schema_name: 'str', assets_dir: 'str', *, baseline_table_name: 'Optional[str]' = None, custom_metrics: 'Optional[List[MonitorMetric]]' = None, data_classification_config: 'Optional[MonitorDataClassificationConfig]' = None, inference_log: 'Optional[MonitorInferenceLog]' = None, latest_monitor_failure_msg: 'Optional[str]' = None, notifications: 'Optional[MonitorNotifications]' = None, schedule: 'Optional[MonitorCronSchedule]' = None, skip_builtin_dashboard: 'Optional[bool]' = None, slicing_exprs: 'Optional[List[str]]' = None, snapshot: 'Optional[MonitorSnapshot]' = None, time_series: 'Optional[MonitorTimeSeries]' = None, warehouse_id: 'Optional[str]' = None) -> 'MonitorInfo' method of databricks.sdk.service.catalog.QualityMonitorsAPI instance
    Creates a new monitor for the specified table.

    The caller must either: 1. be an owner of the table's parent catalog, have **USE_SCHEMA** on th

In [0]:
# ML problem type, either "classification" or "regression"
PROBLEM_TYPE = MonitorInferenceLogProblemType.PROBLEM_TYPE_CLASSIFICATION

# Window sizes to analyze data over
GRANULARITIES = ["1 day"]

username = 'fabieneaulas@gmail.com'
# Directory to store generated dashboard
ASSETS_DIR = f"/Workspace/Users/{username}/databricks_lakehouse_monitoring/model_logs"#?

# Optional parameters
#SLICING_EXPRS = ["Age < 2", "Age > 15", "Sex = 1", "HighChol = 1"]  # Expressions to slice data with


SLICING_EXPRS = ["age < 2", "age > 15", "gender = 1"]  # Expressions to slice data with

In [0]:
print(f"Creating monitor for model_logs")

# Cast Diabetes_binary to long to match labeled_data column type in model_logs
spark.sql(f"""
    CREATE OR REPLACE TABLE {table_name} AS
    SELECT 
        status_code,
        timestamp,
        gender,
        age,
        hypertension,
        heart_disease,
        smoking_history,
        bmi,
        HbA1c_level,
        blood_glucose_level,
        CAST(Diabetes_binary AS LONG) AS Diabetes_binary,
        model_id,
        labeled_data
    FROM {table_name}
""")

# Apply label encoding to match model_logs schema (same as LabelEncoder does: alphabetical order, 0-indexed)
spark.sql(f"""
    CREATE OR REPLACE TABLE {baseline_table_name} AS
    SELECT 
        CAST(CASE 
            WHEN gender = 'Female' THEN 0.0
            WHEN gender = 'Male' THEN 1.0
            WHEN gender = 'Other' THEN 2.0
            ELSE NULL
        END AS DOUBLE) AS gender,
        age,
        hypertension,
        heart_disease,
        CAST(CASE 
            WHEN smoking_history = 'No Info' THEN 0.0
            WHEN smoking_history = 'current' THEN 1.0
            WHEN smoking_history = 'ever' THEN 2.0
            WHEN smoking_history = 'former' THEN 3.0
            WHEN smoking_history = 'never' THEN 4.0
            WHEN smoking_history = 'not current' THEN 5.0
            ELSE NULL
        END AS DOUBLE) AS smoking_history,
        bmi,
        HbA1c_level,
        blood_glucose_level,
        Diabetes_binary,
        id,
        model_id,
        CAST(labeled_data AS LONG) AS labeled_data
    FROM {baseline_table_name}
""")
# para quando for rodar mais de uma vez
try:
    w.quality_monitors.delete(table_name=table_name)
    print(f"Deleted existing monitor for {table_name}")
except Exception as e:
    print(f"No existing monitor to delete: {e}")

info = w.quality_monitors.create(
    table_name=table_name,
    inference_log=MonitorInferenceLog(
        timestamp_col='timestamp',
        granularities=GRANULARITIES,
        model_id_col='model_id',
        prediction_col='Diabetes_binary',
        problem_type=PROBLEM_TYPE,
        label_col='labeled_data'
    ),
    baseline_table_name=baseline_table_name,
    slicing_exprs=SLICING_EXPRS,
    output_schema_name=f"{catalog_name}.{schema_name}",
    assets_dir=ASSETS_DIR
)

Creating monitor for model_logs
Deleted existing monitor for workspace.default.model_logs


In [0]:
import time

# Wait for monitor to be created
while info.status == MonitorInfoStatus.MONITOR_STATUS_PENDING:
    info = w.quality_monitors.get(table_name=table_name)
    time.sleep(10)

assert info.status == MonitorInfoStatus.MONITOR_STATUS_ACTIVE, "Error creating monitor"

In [0]:
table_name

'workspace.default.model_logs'

In [0]:
%sql
-- Check model_logs schema
DESCRIBE workspace.default.model_logs

col_name,data_type,comment
status_code,int,null
timestamp,timestamp,null
gender,double,null
age,double,null
hypertension,bigint,null
heart_disease,bigint,null
smoking_history,double,null
bmi,double,null
HbA1c_level,double,null
blood_glucose_level,bigint,null


In [0]:
%sql
-- Check baseline_features schema
DESCRIBE workspace.default.baseline_features

col_name,data_type,comment
gender,double,null
age,double,null
hypertension,bigint,null
heart_disease,bigint,null
smoking_history,double,null
bmi,double,null
HbA1c_level,double,null
blood_glucose_level,bigint,null
Diabetes_binary,bigint,null
id,bigint,null


In [0]:
# A metric refresh will automatically be triggered on creation
refreshes = w.quality_monitors.list_refreshes(table_name=table_name).refreshes
assert(len(refreshes) > 0)

run_info = refreshes[0]
while run_info.state in (MonitorRefreshInfoState.PENDING, MonitorRefreshInfoState.RUNNING):
    run_info = w.quality_monitors.get_refresh(table_name=table_name, refresh_id=run_info.refresh_id)
    time.sleep(30)

if run_info.state == MonitorRefreshInfoState.FAILED:
    print(f"Monitor refresh failed with message:\n{run_info.message}")
    raise AssertionError(f"Monitor refresh failed: {run_info.message}")

assert run_info.state == MonitorRefreshInfoState.SUCCESS, "Monitor refresh failed"

In [0]:
w.quality_monitors.get(table_name=table_name)

MonitorInfo(output_schema_name='workspace.default', table_name='workspace.default.model_logs', status=<MonitorInfoStatus.MONITOR_STATUS_ACTIVE: 'MONITOR_STATUS_ACTIVE'>, profile_metrics_table_name='workspace.default.model_logs_profile_metrics', drift_metrics_table_name='workspace.default.model_logs_drift_metrics', monitor_version=0, assets_dir='/Workspace/Users/fabieneaulas@gmail.com/databricks_lakehouse_monitoring/model_logs', baseline_table_name='workspace.default.baseline_features', custom_metrics=[], dashboard_id='01f1944d82aa1e9a972771a3b23aa2c5', data_classification_config=None, inference_log=MonitorInferenceLog(problem_type=<MonitorInferenceLogProblemType.PROBLEM_TYPE_CLASSIFICATION: 'PROBLEM_TYPE_CLASSIFICATION'>, timestamp_col='timestamp', granularities=['1 day'], prediction_col='Diabetes_binary', model_id_col='model_id', label_col='labeled_data', prediction_proba_col=None), latest_monitor_failure_msg=None, notifications=None, schedule=None, slicing_exprs=['age < 2', 'age > 15

Meu

![image_1786307078942.png](./image_1786307078942.png "image_1786307078942.png")

Aula databricks

![image_1786307101259.png](./image_1786307101259.png "image_1786307101259.png")

Click the highlighted Dashboard link in the cell output to open the dashboard. You can also navigate to the dashboard from the Catalog Explorer UI.

In [0]:
# Extract workspace URL
workspace_url = spark.conf.get('spark.databricks.workspaceUrl')

# Construct the monitor dashboard URL
monitor_dashboard_url = f"https://{workspace_url}/explore/data/{catalog_name}/{schema_name}/model_logs?o={schema_name}&activeTab=quality"

print(f"Monitor Dashboard URL: {monitor_dashboard_url}")

Monitor Dashboard URL: https://dbc-0b69ef93-5247.cloud.databricks.com/explore/data/workspace/default/model_logs?o=default&activeTab=quality


# Inspect the Metrics Tables

By default, the metrics tables are saved in the default database.

The `create_monitor` call created two new tables: the profile metrics table and the drift metrics table.

# Inspect the Metrics Tables

By default, the metrics tables are saved in the default database.

The `create_monitor` call created two new tables: the profile metrics table and the drift metrics table.

These two tables record the outputs of analysis jobs. The tables use the same name as the primary table to be monitored, with the suffixes `_profile_metrics` and `_drift_metrics`.

## Orientation to the Profile Metrics Table

The profile metrics table has the suffix `_profile_metrics`. For a list of statistics that are shown in the table, see the documentation ([AWS](AWS)|[Azure](Azure)).

* For every column in the primary table, the profile table shows summary statistics for the baseline table and for the primary table. The column `log_type` shows `INPUT` to indicate statistics for the primary table, and `BASELINE` to indicate statistics for the baseline table. The column from the primary table is identified in the column `column_name`.
* For TimeSeries type analysis, the granularity column shows the granularity corresponding to the row. For baseline table statistics, the granularity column shows `null`.
* The table shows statistics for each value of each slice key in each time window, and for the table as whole. Statistics for the table as a whole are indicated by `slice_key = slice_value = null`.
* In the primary table, the `window` column shows the time window corresponding to that row. For baseline table statistics, the `window` column shows `null`.
* Some statistics are calculated based on the table as a whole, not on a single column. In the column `column_name`, these statistics are identified by `:table`.

In [0]:
# Display profile metrics table
profile_table = f"{catalog_name}.{schema_name}.model_logs_profile_metrics"
profile_df = spark.sql(f"SELECT * FROM {profile_table}")
display(profile_df.orderBy(F.rand()).limit(10))

window log_type logging_table_commit_version monitor_version granularity slice_key slice_value model_id column_name count data_type num_nulls avg quantiles min max stddev num_zeros num_nan min_length max_length avg_length non_null_columns frequent_items accuracy_score log_loss roc_auc_score confusion_matrix acceptance_rate median distinct_count percent_nan percent_null percent_zeros percent_distinct precision recall f1_score false_positive_rate statistical_parity equal_opportunity predictive_equality predictive_parity null BASELINE 16 0 null age > 15 false 0 labeled_data 9762 bigint 0 0.0022536365498873182 List(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0

## Orientation to the Drift Metrics Table

The drift metrics table has the suffix `_drift_metrics`. For a list of statistics that are shown in the table, see the documentation ([AWS](AWS)|[Azure](Azure)).

* For every column in the primary table, the drift table shows a set of metrics that compare the current values in the table to the values at the time of the previous analysis run and to the baseline table. The column `drift_type` shows `BASELINE` to indicate drift relative to the baseline table, and `CONSECUTIVE` to indicate drift relative to a previous time window. As in the profile table, the column from the primary table is identified in the column `column_name`.
  * At this point, because this is the first run of this monitor, there is no previous window to compare to. So there are no rows where `drift_type` is `CONSECUTIVE`.
* For TimeSeries type analysis, the granularity column shows the granularity corresponding to that row.
* The table shows statistics for each value of each slice key in each time window, and for the table as whole. Statistics for the table as a whole are indicated by `slice_key = slice_value = null`.
* The `window` column shows the time window corresponding to that row. The `window_cmp` column shows the comparison window. If the comparison is to the baseline table, `window_cmp` is `null`.
* Some statistics are calculated based on the table as a whole, not on a single column. In the column `column_name`, these statistics are identified by `:table`.

https://docs.databricks.com/aws/en/data-governance/unity-catalog/data-quality-monitoring/data-profiling/monitor-output


![image_1786316711309.png](./image_1786316711309.png "image_1786316711309.png")

![image_1786316755352.png](./image_1786316755352.png "image_1786316755352.png")

In [0]:
catalog_name

'workspace'

In [0]:
# Display the drift metrics table
drift_table = f"{catalog_name}.{schema_name}.model_logs_drift_metrics"
display(spark.sql(f"SELECT * FROM {drift_table} ORDER BY RAND() LIMIT 10"))

window,granularity,monitor_version,model_id,slice_key,slice_value,column_name,data_type,window_cmp,drift_type,count_delta,avg_delta,percent_null_delta,percent_zeros_delta,percent_distinct_delta,non_null_columns_delta,js_distance,ks_test,wasserstein_distance,population_stability_index,chi_squared_test,tv_distance,l_infinity_distance
"List(2026-08-09T00:00:00.000Z, 2026-08-10T00:00:00.000Z)",1 day,0,0,age > 15,false,:table,null,null,BASELINE,-9757,null,null,null,null,"List(2, 1)",null,null,null,null,null,null,null
"List(2026-08-09T00:00:00.000Z, 2026-08-10T00:00:00.000Z)",1 day,0,0,age > 15,false,age,double,null,BASELINE,-9757,7.654927269002214,0.0,0.0,19.67219832001639,null,null,"List(1.0, 0.0)",7.6555800000000005,16.09877311905007,null,null,null
"List(2026-08-09T00:00:00.000Z, 2026-08-10T00:00:00.000Z)",1 day,0,0,null,null,labeled_data,bigint,null,BASELINE,-9757,-0.0022536365498873182,0.0,0.22536365498872613,19.979512395001024,null,null,"List(0.0025, 1.0)",0.0024999999999999467,0.6274906864629064,null,null,null
"List(2026-08-09T00:00:00.000Z, 2026-08-10T00:00:00.000Z)",1 day,0,0,age > 15,false,heart_disease,bigint,null,BASELINE,-9757,-3.0731407498463427E-4,0.0,0.03073140749846459,19.979512395001024,null,null,"List(5.0E-4, 1.0)",4.999999999999449E-4,0.6274906864629064,null,null,null
"List(2026-08-09T00:00:00.000Z, 2026-08-10T00:00:00.000Z)",1 day,0,0,age < 2,false,Diabetes_binary,bigint,null,BASELINE,-7656,0.9971283122307792,0.0,-99.71283122307793,19.973893747552538,null,null,"List(0.9975, 1.9531249999997918E-13)",0.9975,11.735082290197923,null,null,null
"List(2026-08-09T00:00:00.000Z, 2026-08-10T00:00:00.000Z)",1 day,0,0,age < 2,false,bmi,double,null,BASELINE,-7656,13.125397467692824,0.0,0.0,3.070095287821431,null,null,"List(0.9955, 3.690562499999789E-12)",13.221789999999995,8.279369020827833,null,null,null
"List(2026-08-09T00:00:00.000Z, 2026-08-10T00:00:00.000Z)",1 day,0,0,age < 2,false,age,double,null,BASELINE,-7656,6.751729539224645,0.0,0.0,19.895574990210154,null,null,"List(1.0, 0.0)",6.7515,16.09877311905007,null,null,null
"List(2026-08-09T00:00:00.000Z, 2026-08-10T00:00:00.000Z)",1 day,0,0,age > 15,false,labeled_data,bigint,null,BASELINE,-9757,-0.0022536365498873182,0.0,0.22536365498872613,19.979512395001024,null,null,"List(0.0025, 1.0)",0.0024999999999999467,0.6274906864629064,null,null,null
"List(2026-08-09T00:00:00.000Z, 2026-08-10T00:00:00.000Z)",1 day,0,0,null,null,bmi,double,null,BASELINE,-9757,13.428413234992096,0.0,0.0,5.228436795738578,null,null,"List(0.9965, 1.0504374999999213E-12)",13.510239999999998,8.279369020827833,null,null,null
"List(2026-08-09T00:00:00.000Z, 2026-08-10T00:00:00.000Z)",1 day,0,0,gender = 1,false,Diabetes_binary,bigint,null,BASELINE,-4722,0.9970382906706157,0.0,-99.70382906706156,19.95768986672308,null,null,"List(0.9975, 1.9531249999997918E-13)",0.9975,11.735082290197923,null,null,null


## Look at fairness and bias metrics

Fairness and bias metrics are calculated for boolean type slices that were defined. The group defined by `slice_value=true` is considered the protected group ([AWS](AWS)|[Azure](Azure)).

In [0]:
fb_cols = ["window", "model_id", "slice_key", "slice_value", "predictive_parity", "predictive_equality", "equal_opportunity", "statistical_parity"]
fb_metrics_df = profile_df.select(fb_cols).filter(f"column_name = ':table' AND slice_value = 'true'")
display(fb_metrics_df.orderBy(F.rand()).limit(10))

window,model_id,slice_key,slice_value,predictive_parity,predictive_equality,equal_opportunity,statistical_parity
null,0,age < 2,true,"List(Map(0 -> 1.0), Map(0 -> 1.0, 1 -> 1.0), Map(0 -> 0.0, 1 -> -1.0), Map(0 -> 1.0, 1 -> 0.0))","List(Map(0 -> 0.0), Map(0 -> 0.0, 1 -> 0.0), Map(0 -> 0.0, 1 -> 0.0), Map(0 -> NaN, 1 -> NaN))","List(Map(0 -> 1.0), Map(0 -> 1.0, 1 -> 1.0), Map(0 -> 0.0, 1 -> -1.0), Map(0 -> 1.0, 1 -> 0.0))","List(Map(0 -> 1.0), Map(0 -> 0.9971283122307792, 1 -> 0.0028716877692207283), Map(0 -> 0.0028716877692207543, 1 -> -0.0028716877692207283), Map(0 -> 1.0028799581097003, 1 -> 0.0))"
null,0,gender = 1,true,"List(Map(0 -> 1.0, 1 -> 1.0), Map(0 -> 1.0, 1 -> 1.0), Map(0 -> 0.0, 1 -> 0.0), Map(0 -> 1.0, 1 -> 1.0))","List(Map(0 -> 0.0, 1 -> 0.0), Map(0 -> 0.0, 1 -> 0.0), Map(0 -> 0.0, 1 -> 0.0), Map(0 -> NaN, 1 -> NaN))","List(Map(0 -> 1.0, 1 -> 1.0), Map(0 -> 1.0, 1 -> 1.0), Map(0 -> 0.0, 1 -> 0.0), Map(0 -> 1.0, 1 -> 1.0))","List(Map(0 -> 0.9984111221449851, 1 -> 0.0015888778550148957), Map(0 -> 0.9970382906706157, 1 -> 0.0029617093293843877), Map(0 -> 0.001372831474369396, 1 -> -0.001372831474369492), Map(0 -> 1.001376909480022, 1 -> 0.5364732586182437))"


### Option 2: Using the UI

If you prefer a graphical interface for setup, Databricks offers a user-friendly UI that guides you step-by-step to create and configure the inference monitor. This method is straightforward and ideal for those who wish to quickly set up monitoring without writing code.

In [0]:
print(f'Your catalog name is: {catalog_name}')
print(f'Your schema name is: {schema_name}')
print(f'Your baseline table name is: {catalog_name}.{schema_name}.baseline_features')

In [0]:
# página 58